# See stimuli from the original dataset 

In [1]:
import altair as alt 
import pandas as pd 
import polars as pl

## Experiment 1

- **Only looked at line charts** 
- “We used a 2 (variability upper vs. lower `flip`) × 2 (more vs. less variability `noise`?) within-subjects design for a total of 4 stimuli types of interest.”
- Used 12 seeds $\times$ 4 stimuli -> 48 stimuli data set
- All participant was shown all 48 images 

In [ ]:
df_1 = pl.read_json("../../data/moritz/stimuli/exp1-stimuli.json")
df_1.head(5)

# # pandas way 
# df_1 = pd.read_json("../../data/moritz/stimuli/exp1-stimuli.json")
# df_1.head(5)

In [ ]:
# 4 stimuli type of interest 
df_1.group_by(["noise", "flip"]).len()

The three columns, `data`, `lineData` and `highNoiseData` is quite confusing. Here's how to understand them: 

- When `noise == [0.0, 0.4]`, i.e., high noise, `data == lineData == highNoiseData`
- When `noise == [0.0, 0.15]`, i.e., low noise, `data = lineData != highNoiseData`

So to recreate Experiment 1 from Mortiz, one only needs data from `data` or `lineData`. 

- [ ] The next step is to figure out programmatically how to do the Vega-Embed + slider + new experiment thingy with ReVISit ... 

Define custom plotting data for row with index `i`: 

In [ ]:
# define a function that generates things 
from typing import Literal 

SimulationType = ["data", "lineData", "highNoiseData"]
PlotType = ["point", "line"]

def get_plot_df(index: int):
    return df_1.filter(pl.col("index") == index).select("data").explode("data").unnest("data")

def plot(index: int, plot_type: PlotType, col: SimulationType): 
    plot_df = df_1.filter(pl.col("index") == index).select(col).explode(col).unnest(col)
    if plot_type == "point": 
        p = alt.Chart(plot_df).mark_point(filled=True).encode(
            x=alt.X("x", axis=None),
            y=alt.Y("y", axis=None).scale(domain=[0, 1])
        ).properties(
            width=500,
            height=200
        )
    elif plot_type == "line":
        p = alt.Chart(plot_df).mark_line().encode(
            x=alt.X("x", axis=None),
            y=alt.Y("y", axis=None).scale(domain=[0, 1])
        ).properties(
            width=500,
            height=200
        )
    return p

In [ ]:
# try a strip plot to see distribution of y values 

alt.Chart(get_plot_df(0)).mark_tick().encode(x = "y")

In [ ]:
# try a strip plot to see y distribution 

alt.Chart(get_plot_df(5)).mark_tick().encode(x = "y")

In [ ]:
df_1_p = df_1.select(["index", "seed", "noise", "flip", "data" ]).with_columns(
    pl.col("data").list.eval(pl.element().struct.field("y")).list.mean().alias("mean_y")
)
df_1_p

In [ ]:
alt.Chart(df_1_p.select("mean_y")).transform_density(
    'mean_y',
    as_=['mean_y', 'density'],
).mark_area().encode(
    x=alt.X("mean_y").scale(domain=[0, 1]),
    y='density:Q',
)

In [ ]:
plot(0, "line", "data")

In [ ]:
plot(1, "line", "data")

In [ ]:
plot(2, "line", "data")

In [ ]:
plot(0, "line", "highNoiseData")

In [ ]:
plot(1, "line", "data")

In [ ]:
plot(1, "line", "lineData")

In [ ]:
plot(1, "line", "highNoiseData")

In [ ]:
plot(2, "line", "data")

In [ ]:
plot(2, "line", "lineData")

In [ ]:
plot(2, "line", "highNoiseData")

In [ ]:
plot(2, "point", "highNoiseData")

In [ ]:
plot(1, "point", "highNoiseData")

Somehow this is making me think about gradience and the judgment of variance / mean ... 

## Experiment 2 

- “but we encoded the data using 1) Cartesian spaced points, 2) points equally spaced along the arc of the line, and 3) the same line encoding used in Experiment 1 (see Figure 4)”
- “We used a 3 `type` (point along x, point along arc, line) × 2 `flip` (variability upper vs. lower) × 2 `noise` (more variability vs. no variability) design.”
- 12 seeds
- each participant viewed 48 images from one plot type 

In [53]:
df_2 = pl.read_json("../../data/moritz/stimuli/exp2-stimuli.json")
df_2.head(12)

# we only want data that is of type point or point_arc 
point_df = df_2.filter(pl.col("type") != "line")

In [55]:
point_df.group_by(["noise", "flip", "type"]).len().sort(["noise", "flip", "type"])

noise,flip,type,len
list[f64],bool,str,u32
"[0.0, 0.0]",false,"""point""",12
"[0.0, 0.0]",false,"""point_arc""",12
"[0.0, 0.0]",true,"""point""",12
"[0.0, 0.0]",true,"""point_arc""",12
"[0.0, 0.4]",false,"""point""",12
"[0.0, 0.4]",false,"""point_arc""",12
"[0.0, 0.4]",true,"""point""",12
"[0.0, 0.4]",true,"""point_arc""",12


In [45]:
def get_plot_df(df: pl.DataFrame, index: int, col: str):
    return df.filter(pl.col("index") == index).select(col).explode(col).unnest(col)

In [67]:
point_df.head(12)

# get_plot_df(point_df, 7, "data")

point_df.to_pandas().head(20)

,index,seed,noise,flip,type,data,lineData,highNoiseData
0,4,1,"[0.0, 0.0]",False,point,"[{'x': 0.0, 'y': 0.8303027032037189}, {'x': 2....","[{'x': 0, 'y': 0.8303027032037189}, {'x': 2, '...","[{'x': 0, 'y': 0.8121457281461196}, {'x': 2, '..."
1,5,1,"[0.0, 0.0]",True,point,"[{'x': 0.0, 'y': 0.16969729679628098}, {'x': 2...","[{'x': 0, 'y': 0.16969729679628098}, {'x': 2, ...","[{'x': 0, 'y': 0.18785427185388037}, {'x': 2, ..."
2,6,1,"[0.0, 0.4]",False,point,"[{'x': 0.0, 'y': 0.8121457281461196}, {'x': 2....","[{'x': 0, 'y': 0.8121457281461196}, {'x': 2, '...","[{'x': 0, 'y': 0.8121457281461196}, {'x': 2, '..."
3,7,1,"[0.0, 0.4]",True,point,"[{'x': 0.0, 'y': 0.18785427185388037}, {'x': 2...","[{'x': 0, 'y': 0.18785427185388037}, {'x': 2, ...","[{'x': 0, 'y': 0.18785427185388037}, {'x': 2, ..."
4,8,1,"[0.0, 0.0]",False,point_arc,"[{'x': 1.9836724189976287, 'y': 0.793659595318...","[{'x': 0, 'y': 0.8303027032037189}, {'x': 2, '...","[{'x': 0, 'y': 0.8121457281461196}, {'x': 2, '..."
5,9,1,"[0.0, 0.0]",True,point_arc,"[{'x': 1.9836724189976274, 'y': 0.206340404681...","[{'x': 0, 'y': 0.16969729679628098}, {'x': 2, ...","[{'x': 0, 'y': 0.18785427185388037}, {'x': 2, ..."
6,10,1,"[0.0, 0.4]",False,point_arc,"[{'x': 2.3481267525147165, 'y': 0.721997997309...","[{'x': 0, 'y': 0.8121457281461196}, {'x': 2, '...","[{'x': 0, 'y': 0.8121457281461196}, {'x': 2, '..."
7,11,1,"[0.0, 0.4]",True,point_arc,"[{'x': 2.3481267525147187, 'y': 0.278002002690...","[{'x': 0, 'y': 0.18785427185388037}, {'x': 2, ...","[{'x': 0, 'y': 0.18785427185388037}, {'x': 2, ..."
8,16,2,"[0.0, 0.0]",False,point,"[{'x': 0.0, 'y': 0.7947001154799134}, {'x': 2....","[{'x': 0, 'y': 0.7947001154799134}, {'x': 2, '...","[{'x': 0, 'y': 0.8710467832469988}, {'x': 2, '..."
9,17,2,"[0.0, 0.0]",True,point,"[{'x': 0.0, 'y': 0.20529988452008652}, {'x': 2...","[{'x': 0, 'y': 0.20529988452008652}, {'x': 2, ...","[{'x': 0, 'y': 0.1289532167530012}, {'x': 2, '..."


Let's compare the mean of when `index == 4` and `index == 8`: 

In [65]:
print(get_plot_df(df_2, 4, "data").select("y").mean())
print(get_plot_df(df_2, 5, "data").select("y").mean())

shape: (1, 1)
┌──────────┐
│ y        │
│ ---      │
│ f64      │
╞══════════╡
│ 0.512701 │
└──────────┘
shape: (1, 1)
┌──────────┐
│ y        │
│ ---      │
│ f64      │
╞══════════╡
│ 0.487299 │
└──────────┘


In [64]:
print(get_plot_df(df_2, 6, "data").select("y").mean())
print(get_plot_df(df_2, 10, "data").select("y").mean())

shape: (1, 1)
┌──────────┐
│ y        │
│ ---      │
│ f64      │
╞══════════╡
│ 0.512701 │
└──────────┘
shape: (1, 1)
┌──────────┐
│ y        │
│ ---      │
│ f64      │
╞══════════╡
│ 0.583157 │
└──────────┘


In [71]:
# when seed is 19 it seems different ... 
point_df.filter(pl.col("seed") == 19)

index,seed,noise,flip,type,data,lineData,highNoiseData
i64,i64,list[f64],bool,str,list[struct[2]],list[struct[2]],list[struct[2]]
136,19,"[0.0, 0.0]",false,"""point""","[{0.0,0.86391}, {2.0,0.750034}, … {118.0,0.194584}]","[{0,0.86391}, {2,0.750034}, … {118,0.194584}]","[{0,1.0}, {2,0.662803}, … {118,0.227795}]"
137,19,"[0.0, 0.0]",true,"""point""","[{0.0,0.13609}, {2.0,0.249966}, … {118.0,0.805416}]","[{0,0.13609}, {2,0.249966}, … {118,0.805416}]","[{0,0.0}, {2,0.337197}, … {118,0.772205}]"
138,19,"[0.0, 0.4]",false,"""point""","[{0.0,1.0}, {2.0,0.662803}, … {118.0,0.227795}]","[{0,1.0}, {2,0.662803}, … {118,0.227795}]","[{0,1.0}, {2,0.662803}, … {118,0.227795}]"
139,19,"[0.0, 0.4]",true,"""point""","[{0.0,0.0}, {2.0,0.337197}, … {118.0,0.772205}]","[{0,0.0}, {2,0.337197}, … {118,0.772205}]","[{0,0.0}, {2,0.337197}, … {118,0.772205}]"
140,19,"[0.0, 0.0]",false,"""point_arc""","[{0.751761,0.821106}, {1.503523,0.778303}, … {118.0,0.194584}]","[{0,0.86391}, {2,0.750034}, … {118,0.194584}]","[{0,1.0}, {2,0.662803}, … {118,0.227795}]"
141,19,"[0.0, 0.0]",true,"""point_arc""","[{0.751761,0.178894}, {1.503523,0.221697}, … {118.0,0.805416}]","[{0,0.13609}, {2,0.249966}, … {118,0.805416}]","[{0,0.0}, {2,0.337197}, … {118,0.772205}]"
142,19,"[0.0, 0.4]",false,"""point_arc""","[{0.392867,0.933763}, {0.785734,0.867526}, … {118.0,0.227795}]","[{0,1.0}, {2,0.662803}, … {118,0.227795}]","[{0,1.0}, {2,0.662803}, … {118,0.227795}]"
143,19,"[0.0, 0.4]",true,"""point_arc""","[{0.392867,0.066237}, {0.785734,0.132474}, … {118.0,0.772205}]","[{0,0.0}, {2,0.337197}, … {118,0.772205}]","[{0,0.0}, {2,0.337197}, … {118,0.772205}]"


In [76]:
alt.Chart(get_plot_df(df_2, 138, "data") ).mark_point(filled=True).encode(
        x=alt.X("x", axis=None),
        y=alt.Y("y", axis=None).scale(domain=[0, 1])
    ).properties(
        width=500,
        height=200
    )

alt.Chart(...)

In [77]:
alt.Chart(get_plot_df(df_2, 142, "data") ).mark_point(filled=True).encode(
        x=alt.X("x", axis=None),
        y=alt.Y("y", axis=None).scale(domain=[0, 1])
    ).properties(
        width=500,
        height=200
    )

alt.Chart(...)

Things are more obvious for the ones where variance is high ... 

In [20]:
alt.Chart(get_plot_df(df_2, 10, "data") ).mark_point(filled=True).encode(
        x=alt.X("x", axis=None),
        y=alt.Y("y", axis=None).scale(domain=[0, 1])
    ).properties(
        width=500,
        height=200
    )

alt.Chart(...)

In [ ]:
alt.Chart(get_plot_df(df_2, 10, "data") ).mark_point(filled=True).encode(
        x=alt.X("x", axis=None),
        y=alt.Y("y", axis=None).scale(domain=[0, 1])
    ).properties(
        width=500,
        height=200
    )

Do stimuli with the same seed share the same data set ...? 

In [19]:
df_2.filter(pl.col("seed") == 1)

index,seed,noise,flip,type,data,lineData,highNoiseData
i64,i64,list[f64],bool,str,list[struct[2]],list[struct[2]],list[struct[2]]
0,1,"[0.0, 0.0]",false,"""line""","[{0.0,0.830303}, {2.0,0.793358}, … {118.0,0.014697}]","[{0,0.830303}, {2,0.793358}, … {118,0.014697}]","[{0,0.812146}, {2,0.726833}, … {118,0.011266}]"
1,1,"[0.0, 0.0]",true,"""line""","[{0.0,0.169697}, {2.0,0.206642}, … {118.0,0.985303}]","[{0,0.169697}, {2,0.206642}, … {118,0.985303}]","[{0,0.187854}, {2,0.273167}, … {118,0.988734}]"
2,1,"[0.0, 0.4]",false,"""line""","[{0.0,0.812146}, {2.0,0.726833}, … {118.0,0.011266}]","[{0,0.812146}, {2,0.726833}, … {118,0.011266}]","[{0,0.812146}, {2,0.726833}, … {118,0.011266}]"
3,1,"[0.0, 0.4]",true,"""line""","[{0.0,0.187854}, {2.0,0.273167}, … {118.0,0.988734}]","[{0,0.187854}, {2,0.273167}, … {118,0.988734}]","[{0,0.187854}, {2,0.273167}, … {118,0.988734}]"
4,1,"[0.0, 0.0]",false,"""point""","[{0.0,0.830303}, {2.0,0.793358}, … {118.0,0.014697}]","[{0,0.830303}, {2,0.793358}, … {118,0.014697}]","[{0,0.812146}, {2,0.726833}, … {118,0.011266}]"
…,…,…,…,…,…,…,…
7,1,"[0.0, 0.4]",true,"""point""","[{0.0,0.187854}, {2.0,0.273167}, … {118.0,0.988734}]","[{0,0.187854}, {2,0.273167}, … {118,0.988734}]","[{0,0.187854}, {2,0.273167}, … {118,0.988734}]"
8,1,"[0.0, 0.0]",false,"""point_arc""","[{1.983672,0.79366}, {3.41949,0.750046}, … {118.0,0.014697}]","[{0,0.830303}, {2,0.793358}, … {118,0.014697}]","[{0,0.812146}, {2,0.726833}, … {118,0.011266}]"
9,1,"[0.0, 0.0]",true,"""point_arc""","[{1.983672,0.20634}, {3.41949,0.249954}, … {118.0,0.985303}]","[{0,0.169697}, {2,0.206642}, … {118,0.985303}]","[{0,0.187854}, {2,0.273167}, … {118,0.988734}]"


In [ ]:
alt.Chart(get_plot_df(df_2, 2, "lineData") ).mark_point(filled=True).encode(
        x=alt.X("x", axis=None),
        y=alt.Y("y", axis=None).scale(domain=[0, 1])
    ).properties(
        width=500,
        height=200
    )

alt.Chart(...)

In [29]:
for i in range(12):
    p = alt.Chart(get_plot_df(df_2, i, "highNoiseData") ).mark_point(filled=True).encode(
        x=alt.X("x", axis=None),
        y=alt.Y("y", axis=None).scale(domain=[0, 1])
    ).properties(
        width=500,
        height=200
    )
    p.save(f"{i}.png")

In [34]:
df_2.filter(pl.col("flip")).group_by(["data", "seed"]).len()

data,seed,len
list[struct[2]],i64,u32
"[{2.455919,0.882956}, {4.712322,0.904883}, … {118.0,0.713821}]",3,1
"[{3.737267,0.982925}, {7.488151,0.975923}, … {118.0,0.727514}]",17,1
"[{0.0,0.187854}, {2.0,0.273167}, … {118.0,0.988734}]",1,2
"[{0.0,0.772005}, {2.0,0.783013}, … {118.0,0.937375}]",15,2
"[{0.0,1.0}, {2.0,0.97745}, … {118.0,0.136691}]",14,2
…,…,…
"[{0.0,0.0}, {2.0,0.337197}, … {118.0,0.772205}]",19,2
"[{1.617419,0.715724}, {3.581067,0.619985}, … {118.0,0.988135}]",11,1
"[{0.751761,0.178894}, {1.503523,0.221697}, … {118.0,0.805416}]",19,1


In [41]:
get_plot_df(df_2, 4, "highNoiseData")

x,y
i64,f64
0,0.812146
2,0.726833
4,0.699056
6,0.598875
8,0.77024
…,…
110,0.209353
112,0.136527
114,0.114223
